# Data Visualization -- Module 2, Class 3

In this notebook you will create 5 types of visualizations from the Superstore dataset:

1. Histogram (distribution of Sales)
2. Boxplot (Profit distribution and outliers)
3. Bar chart (Sales by Category)
4. Correlation heatmap
5. Time series (monthly sales trend)

All 5 are pre-built. At the end, you create 2 additional plots on your own.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Consistent styling
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# This block initializes the essential libraries for data analysis and visualization.
# It also configures a global aesthetic style and standardizes plot dimensions to ensure
# all generated charts are consistent, professional, and easy to read throughout the project.

In [ ]:
# Load dataset (same as data prep lab)
url = "https://raw.githubusercontent.com/dsrscientist/dataset1/master/superstore.csv"

try:
    df = pd.read_csv(url, encoding='latin-1')
    print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
except Exception as e:
    print(f"URL failed ({e}). Upload your CSV manually.")
    from google.colab import files
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    df = pd.read_csv(filename, encoding='latin-1')
    print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")

df.head(3)

# This script attempts to load the Superstore dataset directly from a remote URL using a specific encoding to avoid decoding errors.
# It includes a fallback mechanism that prompts the user for a manual file upload if the URL is unreachable.
# Finally, it displays the data dimensions and previews the first three rows to confirm the dataset has been loaded correctly.

URL failed (HTTP Error 404: Not Found). Upload your CSV manually.


In [ ]:
# Quick data prep: convert date columns
# Identify date-like columns and convert
date_cols = [c for c in df.columns if 'date' in c.lower()]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')
    print(f"Converted {col} to datetime")

# Identify key column names (varies by dataset version)
print(f"\nColumns: {list(df.columns)}")

# This block automatically scans all column names for the keyword 'date' and converts them into the proper datetime format for time-based analysis.
# It uses 'errors=coerce' to handle any invalid entries by turning them into null values, ensuring the process doesn't crash.
# Finally, it prints the updated list of columns to help the user verify the dataset structure and identify available features.

---
## 1. Histogram: Distribution of Sales

A histogram shows how values are distributed. Is the data symmetric? Skewed? Are there clusters?

In [ ]:
# Find the sales column (may be 'Sales' or 'sales')
sales_col = [c for c in df.columns if 'sales' in c.lower()][0]

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(df[sales_col], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
ax.set_xlabel('Sales ($)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Sales')
ax.axvline(df[sales_col].mean(), color='red', linestyle='--', label=f'Mean: ${df[sales_col].mean():,.0f}')
ax.axvline(df[sales_col].median(), color='orange', linestyle='--', label=f'Median: ${df[sales_col].median():,.0f}')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Interpretation: The distribution is right-skewed -- most orders are small,")
print(f"with a long tail of high-value orders. Mean > Median confirms the skew.")

# This code visualizes the statistical distribution of sales data using a histogram to identify patterns and outliers.
# It dynamically locates the sales column and plots vertical lines for the mean and median to highlight data skewness.
# The resulting chart confirms that the dataset is right-skewed, meaning the majority of transactions are small with a few exceptionally high-value orders.

---
## 2. Boxplot: Profit Distribution

A boxplot shows the quartiles (25th, 50th, 75th percentile) and outliers.

In [ ]:
profit_col = [c for c in df.columns if 'profit' in c.lower()][0]

fig, ax = plt.subplots(figsize=(8, 6))
bp = ax.boxplot(df[profit_col].dropna(), vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue', color='navy'),
                medianprops=dict(color='red', linewidth=2))
ax.set_ylabel('Profit ($)')
ax.set_title('Boxplot of Profit')
plt.tight_layout()
plt.show()

# Compute IQR for outlier boundaries
Q1 = df[profit_col].quantile(0.25)
Q3 = df[profit_col].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
n_outliers = ((df[profit_col] < lower_bound) | (df[profit_col] > upper_bound)).sum()

print(f"Q1: {Q1:,.2f} | Q3: {Q3:,.2f} | IQR: {IQR:,.2f}")
print(f"Outlier boundaries: [{lower_bound:,.2f}, {upper_bound:,.2f}]")
print(f"Number of outliers: {n_outliers} ({n_outliers/len(df):.1%} of data)")

# This block performs an outlier analysis on profit data by generating a boxplot and calculating Interquartile Range (IQR) boundaries.
# It identifies the central 50% of the data and defines statistical limits to flag extreme values that fall outside the typical range.
# The final summary provides a clear count and percentage of outliers, helping to understand how much of the dataset consists of unusually high or low profit entries.

---
## 3. Bar Chart: Sales by Category

Compare total sales across product categories.

In [ ]:
category_col = [c for c in df.columns if 'category' in c.lower()]
# Use the first match (usually 'Category', not 'Sub-Category')
cat_col = category_col[0] if category_col else None

if cat_col:
    cat_sales = df.groupby(cat_col)[sales_col].sum().sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(cat_sales.index, cat_sales.values, color=['#2196F3', '#4CAF50', '#FF9800'],
                  edgecolor='black')

    # Add value labels on bars
    for bar, val in zip(bars, cat_sales.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
                f'${val:,.0f}', ha='center', va='bottom', fontweight='bold')

    ax.set_xlabel('Category')
    ax.set_ylabel('Total Sales ($)')
    ax.set_title('Total Sales by Category')
    plt.tight_layout()
    plt.show()
else:
    print("No 'Category' column found. Check column names above.")

# This section aggregates total sales by product category and visualizes the results using a ranked bar chart.
# It dynamically detects the category column, calculates the sum for each group, and adds data labels on top
# of each bar for immediate clarity on financial performance. This visualization makes it easy to identify
# which business segments are driving the highest revenue.

---
## 4. Correlation Heatmap

Shows how numerical features relate to each other. Values range from -1 (perfect negative) to +1 (perfect positive).

In [ ]:
# Select numerical columns for correlation
num_df = df.select_dtypes(include=[np.number])

# Drop ID-like columns (Row ID, Postal Code, etc.) that are not meaningful for correlation
id_like = [c for c in num_df.columns if any(kw in c.lower() for kw in ['id', 'postal', 'code', 'zip'])]
num_df = num_df.drop(columns=id_like, errors='ignore')

corr_matrix = num_df.corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Heatmap (Numerical Features)')
plt.tight_layout()
plt.show()

print("Interpretation:")
print("- Look for strong positive correlations (close to +1): features that move together.")
print("- Look for strong negative correlations (close to -1): features that move opposite.")
print("- Remember: correlation does not imply causation.")

# This section calculates the statistical relationship between numerical variables, excluding non-informative columns like IDs or Postal Codes.
# It generates a heatmap to visualize how variables like Sales, Profit, and Discount interact, where colors represent the strength of the connection.
# The analysis helps identify whether features move in the same or opposite directions, providing key insights into the underlying drivers of the business.

---
## 5. Time Series: Monthly Sales Trend

Track how total sales change over time. Look for seasonality and trends.

In [ ]:
# Find the order date column
order_date_col = [c for c in df.columns if 'order' in c.lower() and 'date' in c.lower()]

if order_date_col:
    odate = order_date_col[0]

    # Resample to monthly totals
    monthly_sales = df.set_index(odate)[sales_col].resample('M').sum()

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(monthly_sales.index, monthly_sales.values, color='steelblue',
            linewidth=2, marker='o', markersize=3)
    ax.set_xlabel('Date')
    ax.set_ylabel('Total Sales ($)')
    ax.set_title('Monthly Total Sales Over Time')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

    print("Look for:")
    print("- Overall trend (going up, down, or flat?)")
    print("- Seasonal patterns (spikes at certain times of year?)")
    print("- Anomalies (sudden drops or spikes?)")
else:
    print("No order date column found. Check column names.")

# This code processes time-series data by resampling individual orders into monthly sales aggregates to reveal long-term business performance.
# It generates a line plot that tracks revenue fluctuations over time, making it easy to spot overall growth trends, recurring seasonal cycles,
# and unexpected anomalies. This visualization is essential for forecasting and understanding the historical trajectory of the company's sales.

---
## TODO: Create 2 Additional Plots

Create 2 plots that were NOT shown above. Some ideas:
- Scatter plot of Sales vs Profit
- Bar chart of Sales by Region
- Pie chart of order count by Segment (Consumer, Corporate, Home Office)
- Violin plot or histogram of Discount distribution
- Stacked bar chart by Category and Sub-Category

For each plot:
1. Write the code
2. Add title and axis labels
3. Write 2-3 sentences of interpretation in a markdown cell below the plot

In [ ]:
# TODO: Plot 1
# Scatter plot of Sales vs Profit
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=df, x=sales_col, y=profit_col, alpha=0.5, color='darkgreen', ax=ax)

ax.set_title('Relationship between Sales and Profit')
ax.set_xlabel('Sales ($)')
ax.set_ylabel('Profit ($)')

# Add a horizontal line at zero profit
ax.axhline(0, color='red', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()



*TODO: Write your interpretation of Plot 1 here.
This scatter plot visualizes the correlation between individual orde sales and their resultingprofit. While there is a general positive trend where higher sales often lead to higher profit, the red dashed line highlights numerous high-value orders that actually resulted in a financial loss(negative profit).

In [ ]:
# TODO: Plot 2
# Pie chart of order count by Segment
segment_counts = df['Segment'].value_counts()

fig, ax = plt.subplots(figsize=(8, 8))
ax.pie(segment_counts, labels=segment_counts.index, autopct='%1.1f%%',
       startangle=140, colors=['#66b3ff', '#99ff99', '#ffcc99'],
       explode=(0.05, 0, 0), shadow=True)

ax.set_title('Distribution of Orders by Customer Segment')
plt.show()




*TODO: Write your interpretation of Plot 2 here.*
The pie chart illustrates the breakdown of the customer base across three main segments. The "Customer" segment represents the largest portion of total orders, followed by Corporate and Hmome Office, suggesting that the business's primary revenue driver is individual retail customers.


---
## Reflection

Answer in a text cell below:

1. The Sales histogram is right-skewed. Why does this happen in retail data? What would it mean if it were perfectly normal?
2. You found outliers in Profit. Should you remove them? What information might you lose?
3. If Discount and Profit have a negative correlation, does that mean the company should stop giving discounts?

Answers to Reflection questions:
1. Right-skewed data: In retail, most transactions are for low-cost everyday items, while expensive luxury items are bought less frequently. If it were "normal," it would mean people buy cheap and expensive items at the exact same frequency, which is unrealistic in business.

2. Outliers in Profit: We should not remove them without investigation. They often represent either your most valuable VIP customers or significant operational errors (like huge returns), providing critical business insights.

3. Discount vs Profit: Not necessarily. While discounts reduce immediate profit per item, they can increase total profit by driving a higher volume of sales or clearing old inventory. The goal is to find the "sweet spot" for discounting.